In [6]:
# Cell 1 - Import & Config

import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_ROOT = Path("../data/mimii/fan/id_00")
SAMPLE_RATE = 16_000

# Windowing config
WINDOW_SEC = 1.0
HOP_SEC = 0.5 # overlapping
N_MELS = 128
FMAX = 8000

WINDOW_SAMPLES = int(WINDOW_SEC * SAMPLE_RATE) # 16_000
HOP_SAMPLES = int(HOP_SEC * SAMPLE_RATE) #8000

print(f"Window: {WINDOW_SAMPLES} samples ({WINDOW_SEC}s)")
print(f"Window: {HOP_SAMPLES} samples ({HOP_SEC}s)")

Window: 16000 samples (1.0s)
Window: 8000 samples (0.5s)


In [7]:
print(DATA_ROOT.resolve())
print(DATA_ROOT.exists())
print(list((DATA_ROOT / "train/normal").glob("*.wav"))[:3])

/Users/gabrielteuchert/Documents/Codes/KI/Project2/AcousticAnomalyDetector/data/mimii/fan/id_00
True
[PosixPath('../data/mimii/fan/id_00/train/normal/00000059.wav'), PosixPath('../data/mimii/fan/id_00/train/normal/00000071.wav'), PosixPath('../data/mimii/fan/id_00/train/normal/00000717.wav')]


In [8]:
print(f"N_MELS config: {N_MELS}")

S = librosa.feature.melspectrogram(y=np.zeros(WINDOW_SAMPLES), sr=SAMPLE_RATE, n_mels=N_MELS, fmax=FMAX)
print(f"Spectrogram shape: {S.shape}")

N_MELS config: 128
Spectrogram shape: (128, 32)


In [9]:
# Cell 2 - Core function: WAV file -> list of Mel-spectogram windows

def extract_windows(filepath: Path, sr: int = SAMPLE_RATE) -> list[np.ndarray]:
    """
    Load a WAV file and slice it into overlapping 1s windows,
    converting each window into a Mel-spectrogram.

    Returns a list of (N_MELS, time_frames) arrays — one per window.
    """
    y, _ = librosa.load(filepath, sr=sr)

    windows = []
    start = 0
    while start + WINDOW_SAMPLES <= len(y):
        chunk = y[start : start + WINDOW_SAMPLES]

        S = librosa.feature.melspectrogram(
            y=chunk, sr=sr, n_mels=N_MELS, fmax=FMAX
        )
        S_dB = librosa.power_to_db(S, ref=np.max)
        windows.append(S_dB.astype(np.float32))   # float32 

        start += HOP_SAMPLES

    return windows


# Quick sanity check on a single file
test_file = sorted((DATA_ROOT / "train/normal").glob("*.wav"))[0]
sample_windows = extract_windows(test_file)

print(f"File duration: 10s -> {len(sample_windows)} windows")
print(f"Each window shape: {sample_windows[0].shape}")

File duration: 10s -> 19 windows
Each window shape: (128, 32)


In [12]:
# Cell 3 - Build the full dataset: X (features) + y (labels)

def build_dataset(normal_files: list[Path], abnormal_files: list[Path] = None):
    """
    Build X, y arrays from a list of normal (and optionally abnormal) files.
    Label convention: 0 = normal, 1 = normal,
    """

    X, y = [], []

    for f in tqdm(normal_files, desc="Normal files"):
        for window in extract_windows(f):
            X.append(window)
            y.append(0)

    if abnormal_files is not None:
        for f in tqdm(abnormal_files, desc="Abnormal files"):
            for window in extract_windows(f):
                X.append(window)
                y.append(1)

    X = np.stack(X) # shape: (n_samples. n_mels, time_frames)
    y = np.array(y, dtype=np.int32)
    return X, y

# -- Build TRAIN set (normal inly - anomaly detection!) --
train_normal_files = sorted((DATA_ROOT / "train/normal").glob("*.wav"))

X_train, y_train = build_dataset(train_normal_files)

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_train unique values: {np.unique(y_train)}")

Normal files: 100%|█████████████████████████████████████████████████████████████████████████| 860/860 [00:25<00:00, 33.94it/s]



X_train shape: (16340, 128, 32)
y_train shape: (16340,)
y_train unique values: [0]


In [16]:
# Cell 4 - Build TEST set (normal + abnormal)

test_normaL_files = sorted((DATA_ROOT / "test/normal").glob("*.wav"))
test_abnormal_files = sorted((DATA_ROOT / "test/abnormal").glob("*.wav"))

X_test, y_test = build_dataset(test_normaL_files, test_abnormal_files)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

unique, counts = np.unique(y_test, return_counts=True)
print(f"\nKlassenverteilung im Test-Set:")
for label, count in zip(unique, counts):
    name = "normal" if label == 0 else "abnormal"
    print(f" {name} (label={label}): {count} samples ({count/len(y_test)*100:.1f})")

Abnormal files: 100%|███████████████████████████████████████████████████████████████████████| 407/407 [00:11<00:00, 34.97it/s]


X_test shape: (10602, 128, 32)
y_test shape: (10602,)

Klassenverteilung im Test-Set:
 normal (label=0): 2869 samples (27.1)
 abnormal (label=1): 7733 samples (72.9)


In [18]:
#Cell 5 - Normalisierung

train_min = X_train.min()
train_max = X_train.max()

print(f"Train range before normalization: [{train_min:.1f}, {train_max:.1f}] dB")

def normalize(X, min_val, max_val):
    return (X - min_val) / (max_val - min_val)

X_train_norm = normalize(X_train, train_min, train_max)
X_test_norm = normalize(X_test, train_min, train_max)

print(f"Train range after: [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]")
print(f"Train range after: [{X_test_norm.min():.3f}, {X_test_norm.max():.3f}]")

Train range before normalization: [-72.9, 0.0] dB
Train range after: [0.000, 1.000]
Train range after: [-0.069, 1.000]


In [20]:
# Cell 6 - Save Data

SAVE_DIR = Path("../data/processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

np.save(SAVE_DIR / "X_train.npy", X_train_norm)
np.save(SAVE_DIR / "y_train.npy", y_train)
np.save(SAVE_DIR / "X_test.npy", X_test_norm)
np.save(SAVE_DIR / "y_test.npy", y_test)

np.save(SAVE_DIR / "norm_params.npy", np.array([train_min, train_max]))

print("Gespeichert in:", SAVE_DIR.resolve())
for f in SAVE_DIR.glob("*.npy"):
    print(f" {f.name}: {f.stat().st_size / 1e6:.1f} MB")

Gespeichert in: /Users/gabrielteuchert/Documents/Codes/KI/Project2/AcousticAnomalyDetector/data/processed
 y_train.npy: 0.1 MB
 y_test.npy: 0.0 MB
 X_test.npy: 173.7 MB
 norm_params.npy: 0.0 MB
 X_train.npy: 267.7 MB


In [21]:
import numpy as np
from pathlib import Path

SAVE_DIR = Path("../data/processed")
X_train = np.load(SAVE_DIR / "X_train.npy")
y_train = np.load(SAVE_DIR / "y_train.npy")

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_train unique: {np.unique(y_train)}")

X_train shape: (16340, 128, 32)
y_train shape: (16340,)
y_train unique: [0]
